In [ ]:
#!pip install -q lxml
#!pip install unidecode
import pandas as pd
import numpy as np
#import geopandas as gpd
import urllib#pour récupérer les données
import bs4#pour rendre lisibles les données
import lxml
import re
import time
from unidecode import unidecode
import urllib

from urllib import request

In [ ]:
#senateur = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/test_17_senateurs.csv")
senateur = pd.read_excel("C:/Users/sylva/OneDrive/Bureau/senat/Data/Prosopographie sénateurs.xlsx")


In [ ]:
senateur = senateur.dropna(how = 'all')

In [ ]:
years = list(range(1789, 1816))

In [ ]:
senateur["naiss"].apply(str)

In [ ]:
def to_date_naiss(x):
    if x == x:
        return float("17"+str(x)[-2:])
    else:
        return None
def to_date_nomin(x):
    if x == x:
        if float(str(x)[2:4])>50:
          return float("17"+str(x)[2:4])
        else:
          return float("18"+str(x)[2:4])
    else:
        return None
def to_date_deces(x):
    if x == x:
        return float("18"+str(x)[2:4])
    else:
        return None

In [ ]:
senateur["date nomination"]

In [ ]:
senateur["annee naiss"] = senateur["naiss"].apply(to_date_naiss)
senateur["annee nomin"] = senateur["date nomination"].apply(to_date_nomin)
senateur["annee deces"] = senateur["date mort"].apply(to_date_deces)

In [ ]:
senateur[["annee naiss", "annee nomin", "annee deces"]]

In [ ]:
def set_place_at_year(personne, year=1800):
    if year < 1789:
        period = "AR"
        nb_period_max = 2
    if 1788 < year < 1799:
        period = "revolution"
        nb_period_max = 10
    if 1798 < year < 1805:
        period = "consulat"
        nb_period_max = 5
    if 1804 < year < 1816:
        period = "empire"
        nb_period_max = 7
    for num_period in range(1, nb_period_max):
        date_num = personne["date "+period+" "+str(num_period)]
        if date_num == str(date_num):
            if str(year) in [str(date) for date in date_num.replace("/", "-").split("-")]:
                date_num = year
            else:
                date_num = 0
        if pd.notna(date_num) and int(date_num)==int(year):
            lieu = personne["lieu "+period+" "+str(num_period)]
            if (type(lieu)==str) : #& (lieu != "paris")
                return personne["lieu "+period+" "+str(num_period)].title().rstrip()
    return ""

In [ ]:
def keep_place_at_year(personne, year):
    if personne[year] != '':
        return personne[year]
    elif year <= personne['annee deces']:
        lieu_actuel = personne['ville naissance']
        for annee in range(1750,year):
            if annee in personne.index:
                if personne[annee]!='':
                    lieu_actuel = personne[annee]
        return lieu_actuel
    else:
        return ''

In [ ]:
for year in years:
    senateur[year] = senateur.apply(set_place_at_year, axis = 1, args = [year])
    senateur[year] = senateur.apply(keep_place_at_year, axis = 1, args = [year])

In [ ]:
senateur

In [ ]:
#senateur["position_sociale"]=senateur["1. place hiérarchie sociale famille"]

In [ ]:
#senateur["position_sociale"].unique()

In [ ]:
"""def to_positions_princ(x):
    if pd.isna(x):
        return ''
    elif x.startswith('admin'):
        return 'admin'
    elif x in ['armée', 'haute armée']:
        return 'militaire'
    elif x in ['paysan', 'artisan', 'cultivateur']:
        return 'travailleur manuel'
    elif x.startswith('médec') or x.startswith('medec'):
        return 'medecin'
    else:
        return x"""

In [ ]:
#senateur["position_sociale_restreint"] = senateur["position_sociale"].apply(to_positions_princ)

In [ ]:
years
for year in years:
    #senateur = pd.merge(senateur, DF_tot2, left_on = year, right_on = "Nom en francais", how = 'left')
    senateur["nom_local"+str(year)] = senateur[year]
#"position_sociale_restreint",

In [ ]:
senateur_for_map = senateur[["nom_local"+str(year) for year in years]+["annee nomin", "nom", "annee naiss", "ville naissance", "annee deces"]]

In [ ]:
Noms_speciaux = {"Allemagne":"Berlin", 'Étranger': "Paris", 'Provence': 'Aix-en-Provence',
                "Sur Le Mein": "Sur le Main", "Sambre Et Mesuse": "Namur", "Sambre Et Meuse": "Namur",
                "Aix": 'Aix-en-Provence', 'Campagne': 'Paris', 'Cassel (All)': 'Cassel, Allemagne',
                'Austerlitz' : 'Brno', 'Frnace' : 'France', 'Eylau': 'Bagrationovsk', 'Ratisbone':'Ratisbonne',
                'Alsace' : 'Strasbourg', 'Ouest': 'Ouest de la France', 'Languedoc':'Toulouse',
                'Uk': 'Angleterre', 'Midi':'Sud de la France', 'Marengo' : 'Alexandrie (Piémont)',
                 'Lorraine':'Metz'
                }
#Df_special = pd.DataFrame.from_dict(Noms_speciaux, orient = "index").reset_index()
#Df_special.columns = ["Nom en francais", "Nom local"]


In [ ]:
def change_by_dic(x):
    if x in Noms_speciaux.keys():
        return Noms_speciaux[x]
    else:
        return x
for year in years:
    senateur_for_map["nom_local"+str(year)] = senateur_for_map["nom_local"+str(year)].apply(change_by_dic)

In [ ]:
senateur_for_map.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")

In [ ]:
for col in senateur_for_map:
    if "Zürich" in senateur_for_map[col].to_list():
        print(col)

In [ ]:
senateur_for_map.columns